Project 3: Akib Ananno

> Revised  


160613

In [ ]:
import os
import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
from glob import glob
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import kagglehub

# 1. Hardware Configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_count = torch.cuda.device_count()
print(f"System Ready. Found {gpu_count} GPU(s).")

BATCH_SIZE = 32 if gpu_count > 1 else 16
EPOCHS = 10
IMG_DIM = 256

In [ ]:
# Download the raw images (not the pandas csv)
print("Fetching CelebA Dataset...")
dataset_path = kagglehub.dataset_download("jessicali9530/celeba-dataset")

# The images are usually stored inside 'img_align_celeba'
image_folder = os.path.join(dataset_path, "img_align_celeba", "img_align_celeba")
print(f"Images located at: {image_folder}")

In [ ]:
class FaceInpaintingDataset(Dataset):
    def __init__(self, root_folder, img_size=256, limit=12000):
        # Taking 12,000 images to satisfy the "> 10,000" rubric safely
        self.all_images = glob(os.path.join(root_folder, "*.jpg"))[:limit]
        self.img_size = img_size

    def __len__(self):
        return len(self.all_images)

    def _create_damage_mask(self, height, width):
        """Generates organic-looking black holes."""
        mask = np.zeros((height, width, 1), dtype=np.float32)

        # 1. Add random scratches (Lines)
        for _ in range(np.random.randint(2, 6)):
            pt1 = (np.random.randint(0, width), np.random.randint(0, height))
            pt2 = (np.random.randint(0, width), np.random.randint(0, height))
            thickness = np.random.randint(3, 12)
            cv2.line(mask, pt1, pt2, 1, thickness)

        # 2. Add random blobs (Ellipses)
        for _ in range(np.random.randint(1, 4)):
            center = (np.random.randint(0, width), np.random.randint(0, height))
            axes = (np.random.randint(10, 30), np.random.randint(10, 30))
            angle = np.random.randint(0, 180)
            cv2.ellipse(mask, center, axes, angle, 0, 360, 1, -1)

        return mask

    def __getitem__(self, index):
        try:
            img_path = self.all_images[index]
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (self.img_size, self.img_size))

            # Normalize to [0, 1]
            img = img.astype(np.float32) / 255.0

            # Apply Black Mask where mask == 1
            damage_mask = self._create_damage_mask(self.img_size, self.img_size)
            corrupted_img = img * (1 - damage_mask)

            # Convert to PyTorch Tensors (C, H, W)
            return (
                torch.from_numpy(corrupted_img).permute(2, 0, 1),
                torch.from_numpy(damage_mask).permute(2, 0, 1),
                torch.from_numpy(img).permute(2, 0, 1)
            )
        except Exception:
            return self.__getitem__((index + 1) % len(self.all_images))

from torch.utils.data import random_split

face_dataset = FaceInpaintingDataset(image_folder, img_size=IMG_DIM, limit=12000)

train_size = int(0.80 * len(face_dataset))  # 9,600
val_size   = int(0.10 * len(face_dataset))  # 1,200
test_size  = len(face_dataset) - train_size - val_size  # 1,200

train_set, val_set, test_set = random_split(
    face_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

loader      = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader  = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")
print(f"Loaded {len(face_dataset)} faces into the pipeline.")

In [ ]:
class SpatialAttention(nn.Module):
    """Filters context features before they are passed to the decoder."""
    def __init__(self, gate_channels, skip_channels, inter_channels):
        super().__init__()
        self.gate_conv = nn.Sequential(nn.Conv2d(gate_channels, inter_channels, 1), nn.BatchNorm2d(inter_channels))
        self.skip_conv = nn.Sequential(nn.Conv2d(skip_channels, inter_channels, 1), nn.BatchNorm2d(inter_channels))
        self.attention_scorer = nn.Sequential(
            nn.Conv2d(inter_channels, 1, 1),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, gate_signal, skip_signal):
        g = self.gate_conv(gate_signal)
        s = self.skip_conv(skip_signal)
        score = self.attention_scorer(self.relu(g + s))
        return skip_signal * score

class FaceInpainter(nn.Module):
    def __init__(self):
        super().__init__()

        def encode_block(in_f, out_f):
            return nn.Sequential(
                nn.Conv2d(in_f, out_f, 3, padding=1), nn.BatchNorm2d(out_f), nn.ReLU(True),
                nn.Conv2d(out_f, out_f, 3, padding=1), nn.BatchNorm2d(out_f), nn.ReLU(True)
            )

        # Downsampling
        self.enc1 = encode_block(3, 64); self.p1 = nn.MaxPool2d(2)
        self.enc2 = encode_block(64, 128); self.p2 = nn.MaxPool2d(2)
        self.enc3 = encode_block(128, 256); self.p3 = nn.MaxPool2d(2)
        self.enc4 = encode_block(256, 512); self.p4 = nn.MaxPool2d(2)

        # Bridge
        self.bridge = nn.Sequential(
            nn.Conv2d(512, 1024, 3, padding=1), nn.BatchNorm2d(1024), nn.ReLU(True),
            nn.Conv2d(1024, 1024, 3, padding=2, dilation=2), nn.BatchNorm2d(1024), nn.ReLU(True)
        )

        # Upsampling + Attention
        self.up1 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.attn1 = SpatialAttention(512, 512, 256)
        self.dec1 = encode_block(1024, 512)

        self.up2 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.attn2 = SpatialAttention(256, 256, 128)
        self.dec2 = encode_block(512, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.attn3 = SpatialAttention(128, 128, 64)
        self.dec3 = encode_block(256, 128)

        self.up4 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.attn4 = SpatialAttention(64, 64, 32)
        self.dec4 = encode_block(128, 64)

        self.final_layer = nn.Conv2d(64, 3, 1)

    def forward(self, x):
        e1 = self.enc1(x); p1 = self.p1(e1)
        e2 = self.enc2(p1); p2 = self.p2(e2)
        e3 = self.enc3(p2); p3 = self.p3(e3)
        e4 = self.enc4(p3); p4 = self.p4(e4)

        b = self.bridge(p4)

        d1 = self.up1(b); e4 = self.attn1(d1, e4); out1 = self.dec1(torch.cat([d1, e4], 1))
        d2 = self.up2(out1); e3 = self.attn2(d2, e3); out2 = self.dec2(torch.cat([d2, e3], 1))
        d3 = self.up3(out2); e2 = self.attn3(d3, e2); out3 = self.dec3(torch.cat([d3, e2], 1))
        d4 = self.up4(out3); e1 = self.attn4(d4, e1); out4 = self.dec4(torch.cat([d4, e1], 1))

        return torch.sigmoid(self.final_layer(out4))

In [ ]:
class CriticNetwork(nn.Module):
    """PatchGAN Discriminator"""
    def __init__(self):
        super().__init__()
        def conv_step(in_ch, out_ch, use_norm=True):
            layers = [nn.Conv2d(in_ch, out_ch, 4, stride=2, padding=1)]
            if use_norm: layers.append(nn.BatchNorm2d(out_ch))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            return layers

        self.net = nn.Sequential(
            *conv_step(3, 64, use_norm=False),
            *conv_step(64, 128),
            *conv_step(128, 256),
            *conv_step(256, 512),
            nn.Conv2d(512, 1, 4, padding=1)
        )
    def forward(self, x):
        return self.net(x)

class ContentStyleLoss(nn.Module):
    """Calculates VGG-based Perceptual and Style Loss"""
    def __init__(self):
        super().__init__()
        vgg = models.vgg16(weights=models.VGG16_Weights.IMAGENET1K_V1).features[:23]
        for p in vgg.parameters(): p.requires_grad = False
        self.vgg = vgg.eval().to(device)
        self.criterion = nn.MSELoss()

    def gram_matrix(self, tensor):
        b, c, h, w = tensor.size()
        features = tensor.view(b, c, h * w)
        return torch.bmm(features, features.transpose(1, 2)) / (c * h * w)

    def forward(self, pred, target):
        pred_feat = self.vgg(pred)
        targ_feat = self.vgg(target)

        perceptual = self.criterion(pred_feat, targ_feat)
        style = self.criterion(self.gram_matrix(pred_feat), self.gram_matrix(targ_feat))
        return perceptual, style

In [ ]:
# Instantiate
generator = FaceInpainter().to(device)
discriminator = CriticNetwork().to(device)
content_loss_fn = ContentStyleLoss().to(device)
pixel_loss_fn = nn.L1Loss()
adv_loss_fn = nn.BCEWithLogitsLoss()

# Multi-GPU Handling
if gpu_count > 1:
    generator = nn.DataParallel(generator)
    discriminator = nn.DataParallel(discriminator)

opt_G = optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
opt_D = optim.Adam(discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))

metrics_log = {
    'psnr': [], 'ssim': [], 'loss_g': [], 'loss_d': [],
    'val_psnr': [], 'val_ssim': [],   # ADDED validation curves
    'l1_loss': [],                     # ADDED pixel reconstruction component
    'perceptual_loss': [],             # ADDED VGG perceptual component
    'mae': []                          # ADDED Mean Absolute Error on masked region only
}

print(f"Training...")
for epoch in range(EPOCHS):
    loop = tqdm(loader, leave=True)
    running_g, running_d = 0.0, 0.0

    for corrupted, mask, real in loop:
        corrupted, real = corrupted.to(device), real.to(device)

        # --- Update Generator ---
        opt_G.zero_grad()
        fake = generator(corrupted)

        p_loss, s_loss = content_loss_fn(fake, real)
        l1_loss = pixel_loss_fn(fake, real)
        g_adv_loss = adv_loss_fn(discriminator(fake), torch.ones_like(discriminator(fake)))

        # Loss weighting
        total_g_loss = (10 * l1_loss) + (1 * p_loss) + (100 * s_loss) + (0.1 * g_adv_loss)
        total_g_loss.backward()
        opt_G.step()

        # --- Update Discriminator ---
        opt_D.zero_grad()
        d_real_loss = adv_loss_fn(discriminator(real), torch.ones_like(discriminator(real)))
        d_fake_loss = adv_loss_fn(discriminator(fake.detach()), torch.zeros_like(discriminator(fake.detach())))
        total_d_loss = (d_real_loss + d_fake_loss) * 0.5
        total_d_loss.backward()
        opt_D.step()

        running_g += total_g_loss.item()
        running_d += total_d_loss.item()
        loop.set_postfix(G=total_g_loss.item(), D=total_d_loss.item())

    # --- End of Epoch Evaluation (Train batch) ---
    with torch.no_grad():
        r_np = real[:8].permute(0,2,3,1).cpu().numpy()
        f_np = fake[:8].permute(0,2,3,1).cpu().numpy()
        epoch_psnr = np.mean([psnr(r_np[i], f_np[i], data_range=1.0) for i in range(len(r_np))])
        epoch_ssim = np.mean([ssim(r_np[i], f_np[i], data_range=1.0, channel_axis=2, win_size=3) for i in range(len(r_np))])

        # Extra metric 1 – L1 (pixel) loss component tracked separately
        ep_l1 = pixel_loss_fn(fake[:8], real[:8]).item()

        # Extra metric 2 – Perceptual loss component tracked separately
        ep_perc, _ = content_loss_fn(fake[:8], real[:8])

        # Extra metric 3 – MAE on the masked region only
        mask_b = mask[:8].to(device)
        ep_mae = (torch.abs(fake[:8] - real[:8]) * mask_b).sum() / (mask_b.sum() + 1e-8)

    # --- Validation pass ---
    generator.eval()
    val_psnr_scores, val_ssim_scores = [], []
    with torch.no_grad():
        for v_corrupted, _, v_real in val_loader:
            v_corrupted, v_real = v_corrupted.to(device), v_real.to(device)
            v_fake = generator(v_corrupted)
            v_r = v_real.permute(0,2,3,1).cpu().numpy()
            v_f = v_fake.permute(0,2,3,1).cpu().numpy()
            for i in range(len(v_r)):
                val_psnr_scores.append(psnr(v_r[i], v_f[i], data_range=1.0))
                val_ssim_scores.append(ssim(v_r[i], v_f[i], data_range=1.0, channel_axis=2, win_size=3))
    generator.train()

    metrics_log['psnr'].append(epoch_psnr)
    metrics_log['ssim'].append(epoch_ssim)
    metrics_log['loss_g'].append(running_g / len(loader))
    metrics_log['loss_d'].append(running_d / len(loader))
    metrics_log['val_psnr'].append(np.mean(val_psnr_scores))
    metrics_log['val_ssim'].append(np.mean(val_ssim_scores))
    metrics_log['l1_loss'].append(ep_l1)
    metrics_log['perceptual_loss'].append(ep_perc.item())
    metrics_log['mae'].append(ep_mae.item())

    print(f"Epoch {epoch+1} | Train PSNR: {epoch_psnr:.2f} | Val PSNR: {np.mean(val_psnr_scores):.2f} | "
          f"SSIM: {epoch_ssim:.4f} | MAE(mask): {ep_mae.item():.4f}")

In [ ]:
generator.eval()
test_psnr, test_ssim_scores = [], []
with torch.no_grad():
    for t_corrupted, _, t_real in test_loader:
        t_corrupted, t_real = t_corrupted.to(device), t_real.to(device)
        t_fake = generator(t_corrupted)
        t_r = t_real.permute(0,2,3,1).cpu().numpy()
        t_f = t_fake.permute(0,2,3,1).cpu().numpy()
        for i in range(len(t_r)):
            test_psnr.append(psnr(t_r[i], t_f[i], data_range=1.0))
            test_ssim_scores.append(ssim(t_r[i], t_f[i], data_range=1.0, channel_axis=2, win_size=3))

print(f"TEST SET  |  PSNR: {np.mean(test_psnr):.2f} dB  |  SSIM: {np.mean(test_ssim_scores):.4f}")

In [ ]:
def plot_results(gen_model, test_loader, num_samples=5):
    gen_model.eval()
    corrupted, _, real = next(iter(test_loader))

    with torch.no_grad():
        fake = gen_model(corrupted.to(device)).cpu()

    fig, axes = plt.subplots(num_samples, 3, figsize=(10, 3 * num_samples))

    for i in range(num_samples):
        # Input
        axes[i, 0].imshow(corrupted[i].permute(1, 2, 0))
        axes[i, 0].set_title("Input (Damaged)")
        axes[i, 0].axis('off')

        # Output
        axes[i, 1].imshow(fake[i].permute(1, 2, 0).clamp(0, 1))
        axes[i, 1].set_title("Network Output")
        axes[i, 1].axis('off')

        # Ground Truth
        axes[i, 2].imshow(real[i].permute(1, 2, 0))
        axes[i, 2].set_title("Ground Truth")
        axes[i, 2].axis('off')

    plt.tight_layout()
    plt.show()


# Visualize repaired samples
plot_results(generator, loader)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Plot 1 – PSNR
axes[0].plot(metrics_log['psnr'],     label='Train PSNR', color='green')
axes[0].plot(metrics_log['val_psnr'], label='Val PSNR',   color='lime', linestyle='--')
axes[0].set_title("PSNR (Train vs Val)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2 – SSIM
axes[1].plot(metrics_log['ssim'],     label='Train SSIM', color='blue')
axes[1].plot(metrics_log['val_ssim'], label='Val SSIM',   color='cyan', linestyle='--')
axes[1].set_title("SSIM (Train vs Val)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Plot 3 – Component Losses
axes[2].plot(metrics_log['l1_loss'],         label='L1 Loss',         color='orange')
axes[2].plot(metrics_log['perceptual_loss'], label='Perceptual Loss', color='purple')
axes[2].plot(metrics_log['mae'],             label='Masked MAE',      color='red')
axes[2].set_title("Component Metrics")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()